In [ ]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/facebook/nllb-200-distilled-600M

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/facebook/nllb-200-distilled-600M)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Use a pipeline as a high-level helper
# Warning: Pipeline type "translation" is no longer supported in transformers v5.
# You must load the model directly (see below) or downgrade to v4.x with:
# pip install "transformers<5.0.0"
from transformers import pipeline

pipe = pipeline("translation", model="facebook/nllb-200-distilled-600M")

In [ ]:
# pip install -U transformers accelerate
# Load model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M", device_map="auto")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

...

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
!pip install -q transformers sentencepiece gradio

In [ ]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/nllb-200-distilled-600M"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model = model.to(device)
model.eval()

print("Universal Translator model loaded successfully!")

Using device: cuda


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Universal Translator model loaded successfully!


In [ ]:
languages = {
    "English": "eng_Latn",
    "Tamil": "tam_Taml",
    "Hindi": "hin_Deva",
    "Telugu": "tel_Telu",
    "Malayalam": "mal_Mlym",
    "Kannada": "kan_Knda",
    "Bengali": "ben_Beng",
    "Marathi": "mar_Deva",
    "Gujarati": "guj_Gujr",
    "Punjabi": "pan_Guru",
    "Urdu": "urd_Arab",
    "Arabic": "arb_Arab",
    "French": "fra_Latn",
    "German": "deu_Latn",
    "Spanish": "spa_Latn",
    "Portuguese": "por_Latn",
    "Italian": "ita_Latn",
    "Russian": "rus_Cyrl",
    "Chinese (Simplified)": "zho_Hans",
    "Japanese": "jpn_Jpan",
    "Korean": "kor_Hang",
    "Turkish": "tur_Latn",
    "Indonesian": "ind_Latn",
    "Malay": "zsm_Latn",
    "Thai": "tha_Thai",
    "Vietnamese": "vie_Latn",
    "Swahili": "swh_Latn",
    "Persian": "pes_Arab",
    "Dutch": "nld_Latn",
    "Polish": "pol_Latn",
    "Ukrainian": "ukr_Cyrl",
    "Greek": "ell_Grek",
    "Hebrew": "heb_Hebr",
    "Nepali": "npi_Deva",
    "Sinhala": "sin_Sinh",
    "Odia": "ory_Orya",
    "Assamese": "asm_Beng",
    "Azerbaijani": "azj_Latn",
    "Filipino": "tgl_Latn",
    "Romanian": "ron_Latn",
    "Hungarian": "hun_Latn",
    "Czech": "ces_Latn",
    "Danish": "dan_Latn",
    "Finnish": "fin_Latn",
    "Norwegian": "nob_Latn",
    "Swedish": "swe_Latn",
    "Serbian": "srp_Cyrl",
    "Croatian": "hrv_Latn",
    "Bulgarian": "bul_Cyrl",
    "Amharic": "amh_Ethi",
    "Yoruba": "yor_Latn",
    "Igbo": "ibo_Latn",
    "Hausa": "hau_Latn",
    "Khmer": "khm_Khmr",
    "Lao": "lao_Laoo",
    "Burmese": "mya_Mymr"
}

print("Languages available:", len(languages))

Languages available: 56


In [ ]:
def translate_text(text, source_language, target_language):

    if not text or not text.strip():
        return "Please enter text to translate."

    if source_language == target_language:
        return "Please select two different languages."

    source_code = languages[source_language]
    target_code = languages[target_language]

    tokenizer.src_lang = source_code

    inputs = tokenizer(
        text.strip(),
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        translated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(
                target_code
            ),
            max_length=512,
            num_beams=4
        )

    translated_text = tokenizer.batch_decode(
        translated_tokens,
        skip_special_tokens=True
    )[0]

    return translated_text

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as app:

    gr.Markdown("# 🌍 Universal Language Translator")
    gr.Markdown(
        "Translate text between supported languages using "
        "Hugging Face NLLB-200."
    )

    with gr.Row():
        source_language = gr.Dropdown(
            choices=list(languages.keys()),
            value="English",
            label="Translate From"
        )

        target_language = gr.Dropdown(
            choices=list(languages.keys()),
            value="Tamil",
            label="Translate To"
        )

    input_text = gr.Textbox(
        label="Enter Your Text",
        placeholder="Type or paste your text here...",
        lines=5
    )

    translate_button = gr.Button(
        "🌐 Translate",
        variant="primary"
    )

    output_text = gr.Textbox(
        label="Translated Text",
        lines=5
    )

    translate_button.click(
        fn=translate_text,
        inputs=[
            input_text,
            source_language,
            target_language
        ],
        outputs=output_text
    )

    gr.ClearButton(
        [input_text, output_text],
        value="Clear"
    )

app.launch(share=True)

/tmp/ipykernel_1118/2330715648.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://412156a85a3a7b17fa.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
